# MeterIQ — Smart Meter Data Preprocessing

This notebook is the preprocessing stage after EDA. It uses the observed structure of the supplied archive, keeps the ZIP/raw records unchanged, and writes a separate cleaned CSV plus a quality report.

The cleaning policy is deliberately conservative: exact duplicate rows and records missing a meter, timestamp, or valid non-negative `t_kWh` are separated into a rejected-records file. No interpolation, resampling, or scaling is performed here; those decisions must follow the observed sampling frequency and model requirements.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import json
import re

import pandas as pd

pd.set_option('display.max_columns', None)

In [ ]:
# Update this only if the ZIP is moved. The archive itself is never modified.
ZIP_PATH = Path('/Users/yashitarora/Downloads/archive (7).zip')
OUTPUT_DIR = Path('data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    raise FileNotFoundError(f'Archive not found: {ZIP_PATH}')

with ZipFile(ZIP_PATH) as archive:
    file_info = {item.filename: item.file_size for item in archive.infolist() if item.filename.lower().endswith('.csv')}

pd.DataFrame({'dataset': file_info.keys(), 'uncompressed_size_mb': [round(size / 1_048_576, 1) for size in file_info.values()]})

## Select one file

Begin with the small daily aggregated files. The raw files are much larger, so load them one at a time with sufficient memory. Set `ROWS_TO_READ` to a number only for a dry run; do not use a partial result as the final processed dataset.

In [ ]:
DATASET_NAME = 'SM Cleaned Data BR Aggregated.csv'
ROWS_TO_READ = None  # e.g. 100_000 for a dry run of a large raw file

if DATASET_NAME not in file_info:
    raise ValueError('Choose an exact dataset name shown in the table above.')

with ZipFile(ZIP_PATH) as archive:
    with archive.open(DATASET_NAME) as source:
        raw = pd.read_csv(source, nrows=ROWS_TO_READ, low_memory=False)

print(f'Loaded {len(raw):,} rows from {DATASET_NAME}')
print('Columns:', raw.columns.tolist())
raw.head()

In [ ]:
# Validate the fields actually observed in this archive before cleaning.
# Aggregated files use Date; raw files use x_Timestamp.
TIME_SOURCE = 'Date' if 'Date' in raw.columns else 'x_Timestamp'
REQUIRED_COLUMNS = {'meter', TIME_SOURCE, 't_kWh'}
missing_required = REQUIRED_COLUMNS - set(raw.columns)
if missing_required:
    raise ValueError(f'Unexpected schema. Required columns missing: {sorted(missing_required)}')

OPTIONAL_SENSOR_COLUMNS = [
    name for name in ['z_Avg Voltage (Volt)', 'z_Avg Current (Amp)', 'y_Freq (Hz)']
    if name in raw.columns
]
print('Timestamp source:', TIME_SOURCE)
print('Optional sensor columns:', OPTIONAL_SENSOR_COLUMNS)

In [ ]:
# Create a working copy and standardize names only in the processed output.
# Raw field names and raw values remain available in `raw`.
working = raw.copy()
working['meter_id'] = working['meter'].astype('string').str.strip()
working['timestamp'] = pd.to_datetime(working[TIME_SOURCE], errors='coerce')
working['consumption_kwh'] = pd.to_numeric(working['t_kWh'], errors='coerce')

for source_name in OPTIONAL_SENSOR_COLUMNS:
    clean_name = {
        'z_Avg Voltage (Volt)': 'avg_voltage_volt',
        'z_Avg Current (Amp)': 'avg_current_amp',
        'y_Freq (Hz)': 'frequency_hz',
    }[source_name]
    working[clean_name] = pd.to_numeric(working[source_name], errors='coerce')

working[['meter_id', 'timestamp', 'consumption_kwh']].head()

In [ ]:
# Record reasons before filtering. A record can have more than one reason.
reasons = pd.DataFrame(index=working.index)
reasons['exact_duplicate'] = working.duplicated(keep='first')
reasons['missing_meter_id'] = working['meter_id'].isna() | working['meter_id'].eq('')
reasons['invalid_timestamp'] = working['timestamp'].isna()
reasons['invalid_consumption'] = working['consumption_kwh'].isna()
reasons['negative_consumption'] = working['consumption_kwh'].lt(0).fillna(False)

reject_mask = reasons.any(axis=1)
rejected = working.loc[reject_mask].copy()
rejected['rejection_reasons'] = reasons.loc[reject_mask].apply(
    lambda row: '; '.join(row.index[row].tolist()), axis=1
)
clean = working.loc[~reject_mask].copy().sort_values(['meter_id', 'timestamp']).reset_index(drop=True)

quality_counts = reasons.sum().sort_values(ascending=False).rename('record_count').to_frame()
quality_counts['percent_of_loaded_rows'] = (quality_counts['record_count'] / len(working) * 100).round(3)
print(f'Accepted records: {len(clean):,}')
print(f'Rejected records: {len(rejected):,}')
quality_counts

## Post-cleaning checks

This checks for repeated timestamps per meter but does **not** remove them automatically: multiple readings at the same time may require domain clarification.

In [ ]:
same_time_records = clean.duplicated(['meter_id', 'timestamp'], keep=False).sum()
intervals = clean.groupby('meter_id')['timestamp'].diff().dropna()

print('Unique meters:', clean['meter_id'].nunique())
print('Clean date range:', clean['timestamp'].min(), 'to', clean['timestamp'].max())
print('Records sharing a meter and timestamp:', f'{same_time_records:,}')
print('Median interval:', intervals.median() if not intervals.empty else 'not available')
clean[['meter_id', 'timestamp', 'consumption_kwh']].describe(include='all')

In [ ]:
# Save separate audit-friendly outputs. All files are generated locally; none are pushed to GitHub.
slug = re.sub(r'[^a-z0-9]+', '_', Path(DATASET_NAME).stem.lower()).strip('_')
clean_path = OUTPUT_DIR / f'{slug}_processed.csv'
rejected_path = OUTPUT_DIR / f'{slug}_rejected.csv'
report_path = OUTPUT_DIR / f'{slug}_preprocessing_report.json'

clean.to_csv(clean_path, index=False)
rejected.to_csv(rejected_path, index=False)
report = {
    'source_archive': str(ZIP_PATH),
    'source_member': DATASET_NAME,
    'loaded_rows': int(len(raw)),
    'rows_to_read': ROWS_TO_READ,
    'accepted_rows': int(len(clean)),
    'rejected_rows': int(len(rejected)),
    'quality_counts': {name: int(value) for name, value in reasons.sum().items()},
    'same_meter_timestamp_records': int(same_time_records),
    'timestamp_source': TIME_SOURCE,
    'policy': 'Exact duplicates and rows with missing meter/timestamp/consumption or negative consumption are separated; raw archive is unchanged.',
}
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')

print('Saved:', clean_path)
print('Saved:', rejected_path)
print('Saved:', report_path)

## Before moving to feature engineering

1. Run this notebook once for each selected Mathura and Bareilly file.
2. Review the rejected-records file and quality report.
3. Confirm how to interpret any negative `t_kWh` records and same-meter/same-timestamp readings with the dataset documentation.
4. Use the observed median interval to decide appropriate lag, rolling-window, and resampling features.

The next stage can then build time-aware features without leakage.